Precision, Recall, F1-score
Clase 1 = impago

Precision: De los clientes que el modelo predice como impago, cuántos realmente incumplen.

Recall: De todos los clientes que realmente incumplen, cuántos el modelo predice correctamente.

F1-score: Media armónica entre precision y recall, útil para balancear ambos.


Tu objetivo es detectar impagos (clase 1). Por tanto:

Recall de clase 1 es la métrica más importante
→ quieres minimizar falsos negativos (clientes que incumplen pero tu modelo dice que no).

Precision importa menos que recall si estás dispuesto a aceptar algunos falsos positivos (alertas de riesgo innecesarias).

F1-score de clase 1 te da un balance, útil para comparar modelos.

ROC-AUC es buena métrica general de ranking de riesgo.

In [18]:
# =====================================================
import numpy as np
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    recall_score,
    precision_score,   
    make_scorer,
    silhouette_score,
    precision_recall_curve
)

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import NearestCentroid

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import time

In [ ]:
# 1. CARGAR DATOS
def cargar_y_preparar_datos(ruta_archivo):
    df = pd.read_excel(ruta_archivo)
    # Filtrar solo vivienda y copiar para evitar warnings
    df_viv = df[df['Proposito'].astype(str)
                .str.contains('Vivienda', case=False, na=False)].copy()
    # Label
    df_viv['Impago_Label'] = df_viv['Impago'].map({0:0, 1:1})
    return df_viv

# Ajusta esta ruta si es necesario
ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')

if os.path.exists(ruta_real):
    df = cargar_y_preparar_datos(ruta_real)
else:
    print(f" ATENCIÓN: No se encuentra el archivo en {ruta_real}")
    df = pd.DataFrame() 

In [6]:
# 2. DEFINIR X e y
if not df.empty:
    target_col = "Impago_Label"
    columnas_a_eliminar = ["ID", "Impago", "Prima", "Proposito"]

    y = df[target_col]
    X = df.drop(columns=[target_col])
    X = X.drop(columns=[col for col in columnas_a_eliminar if col in X.columns])

    # Eliminar alta cardinalidad
    high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
    X = X.drop(columns=high_card_cols)

    # One-hot encoding
    cat_cols = X.select_dtypes(include="object").columns
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

    X = X.astype("float32")
    
# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

In [ ]:
# 4. CLUSTERING: TORNEO K-MEANS vs AGLOMERATIVO
print("--- Iniciando Optimización de Clustering ---")

#1. Escalado
scaler_cluster = StandardScaler()
X_train_cluster = scaler_cluster.fit_transform(X_train)
X_test_cluster = scaler_cluster.transform(X_test)

# ENCONTRAR EL NÚMERO DE CLUSTERS (K) ÓPTIMO (Probamos de 2 a 5 clusters y nos quedamos con el mejor)

print(" Buscando el número óptimo de clusters (k)...")
best_k = 3  
best_k_score = -1

for k in [2, 3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_train_cluster)
    score = silhouette_score(X_train_cluster, labels)
    print(f"   k={k} -> Silhouette: {score:.4f}")
        
    if score > best_k_score:
        best_k_score = score
        best_k = k

print(f" Número óptimo seleccionado: k={best_k}")

#PASO 2: TORNEO CON EL K GANADOR (KMeans vs Aglomerativo)
print(f"\n--- Iniciando Torneo (usando k={best_k}) ---")

scores = {}
labels_storage = {}

# Función auxiliar
def asignar_clusters(model, X_train_scaled, X_test_scaled):
    labels_train = model.fit_predict(X_train_scaled)
    if hasattr(model, "predict"):
        labels_test = model.predict(X_test_scaled)
    else:
        centroid_clf = NearestCentroid()
        centroid_clf.fit(X_train_scaled, labels_train)
        labels_test = centroid_clf.predict(X_test_scaled)
        
    return labels_train, labels_test  
# Opción A: KMeans
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
k_labels_train, k_labels_test = asignar_clusters(kmeans, X_train_cluster, X_test_cluster)
scores["KMeans"] = silhouette_score(X_train_cluster, k_labels_train)
labels_storage["KMeans"] = (k_labels_train, k_labels_test)
print(f"Silhouette KMeans: {scores['KMeans']:.4f}")

# Opción B: Aglomerativo
agg = AgglomerativeClustering(n_clusters=best_k)
a_labels_train, a_labels_test = asignar_clusters(agg, X_train_cluster, X_test_cluster)
scores["Agglomerative"] = silhouette_score(X_train_cluster, a_labels_train)
labels_storage["Agglomerative"] = (a_labels_train, a_labels_test)
print(f"Silhouette Agglomerative: {scores['Agglomerative']:.4f}")

#SELECCIÓN FINAL
best_model_name = max(scores, key=scores.get)
print(f"GANADOR DEL TORNEO: {best_model_name} con k={best_k}")

# Aplicar al dataset
final_labels_train, final_labels_test = labels_storage[best_model_name]

# One-Hot Encoding
train_dummies = pd.get_dummies(final_labels_train, prefix='Cluster_Group')
test_dummies = pd.get_dummies(final_labels_test, prefix='Cluster_Group')

# Alinear columnas
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

# Unir
train_dummies.index = X_train.index
test_dummies.index = X_test.index
X_train = pd.concat([X_train, train_dummies], axis=1)
X_test = pd.concat([X_test, test_dummies], axis=1)

print(f" Variables de cluster añadidas. Nuevas columnas: {list(train_dummies.columns)}")


--- Iniciando Optimización de Clustering ---
 Buscando el número óptimo de clusters (k)...
   k=2 -> Silhouette: 0.1469
   k=3 -> Silhouette: 0.1329
   k=4 -> Silhouette: 0.1505
   k=5 -> Silhouette: 0.1456
 Número óptimo seleccionado: k=4

--- Iniciando Torneo (usando k=4) ---
Silhouette KMeans: 0.1505
Silhouette Agglomerative: 0.1801
GANADOR DEL TORNEO: Agglomerative con k=4
 Variables de cluster añadidas. Nuevas columnas: ['Cluster_Group_0', 'Cluster_Group_1', 'Cluster_Group_2', 'Cluster_Group_3']


Cuando estudiamos Clustering en la teoría, se supone que lo mejor es donde los datos forman "islas" separadas (Silhouette > 0.70).

Pero en la vida real (y más en finanzas), los clientes no son islas;sino el el que cobra 1.500€ se mezcla con el que cobra 1.550€. El algoritmo ha cortado esa nube en 4 trozos. Como las fronteras entre los trozos se tocan y se solapan mucho, el Silhouette nos dice: "Oye, los grupos están muy pegados". Y es verdad, pero eso no significa que no sean útiles para tu modelo predictivo.

In [27]:
# EXTRA: PERFILADO DE CLUSTERS (Que tipo de personas hay en cada cluster)
print("RADIOGRAFÍA DE LOS 4 CLUSTERS")

# 1. Juntamos los datos temporalmente para analizarlos
df_perfil = X_train.copy()
df_perfil['Impago_Real'] = y_train
df_perfil['Cluster_Asignado'] = final_labels_train  # Del 0 al 3

# 2. Vemos cuánta gente hay y cuántos morosos tiene cada cluster
resumen_clusters = df_perfil.groupby('Cluster_Asignado').agg(
    Total_Clientes=('Impago_Real', 'count'),
    Tasa_Morosidad_Pct=('Impago_Real', lambda x: (x.mean() * 100).round(2))
).reset_index()

print("MOROSIDAD POR CLUSTER:")
print(resumen_clusters.to_string(index=False))

# 3. Vemos cómo es el cliente medio de cada cluster
# Quitamos las columnas dummy de los propios clusters para no ensuciar
cols_a_ignorar = [c for c in df_perfil.columns if "Cluster_Group" in c]
df_perfil_limpio = df_perfil.drop(columns=cols_a_ignorar)

perfil_variables = df_perfil_limpio.groupby('Cluster_Asignado').mean().round(2).T

print("PERFIL MEDIO DEL CLIENTE EN CADA CLUSTER:")
# Mostramos todas las filas para que puedas ver todas las variables
pd.set_option('display.max_rows', None) 
print(perfil_variables)

RADIOGRAFÍA DE LOS 4 CLUSTERS
MOROSIDAD POR CLUSTER:
 Cluster_Asignado  Total_Clientes  Tasa_Morosidad_Pct
                0            3853               11.06
                1            3071               14.00
                2             443                9.03
                3             826                4.96
PERFIL MEDIO DEL CLIENTE EN CADA CLUSTER:
Cluster_Asignado                               0          1          2          3
Num_Creditos                            2.210000   2.200000   2.010000   2.030000
Duracion                               32.549999  33.279999  33.099998  31.209999
Ratio_Deuda_Ingresos                    0.510000   0.490000   0.450000   0.480000
Posesion_Hipoteca                       0.420000   0.430000   0.420000   0.410000
Personas_Cargo                          0.460000   0.460000   0.380000   0.470000
Fiador                                  0.510000   0.500000   0.490000   0.490000
Estudios_Escolar                        0.990000   0.000000  

¡BINGO! 🎉 ¿Ves cómo el Silhouette bajo no importaba? Tu algoritmo de clustering ha funcionado de maravilla y ha encontrado patrones de negocio clarísimos.

Ha segmentado a los clientes usando, casi exclusivamente, su Nivel de Estudios y su Estado Civil. Y lo más fascinante es que ha destapado cosas anti-intuitivas que le van a encantar a cualquier tribunal o jefe de riesgos.

Aquí tienes el perfilado exacto de cada cluster, listo para copiar y pegar en tu informe o TFM

Shutterstock


:

🔴 Cluster 1: "Universitarios Solteros" (EL MAYOR RIESGO)
Tasa de Morosidad: 14.00% (Los que más impagan con diferencia).

Volumen: 3.071 clientes.

¿Quiénes son? El 99% tiene Grado Universitario y el 73% están solteros.

Insight de negocio: Aunque tengan carrera universitaria, este grupo es el más peligroso para el banco. Suelen ser jóvenes independizados o perfiles que, pese a tener estudios, asumen más riesgo del que pueden pagar.

🟡 Cluster 0: "El Perfil Base" (RIESGO MEDIO-ALTO)
Tasa de Morosidad: 11.06%.

Volumen: 3.853 clientes (El grupo más grande).

¿Quiénes son? El 99% solo tiene "Estudios Escolares" (básicos) y tienen el Ratio de Deuda más alto de todos (0.51). Un 54% son solteros.

Insight de negocio: Es el cliente "promedio" trabajador con bajo nivel educativo y alto endeudamiento relativo.

🟢 Cluster 2: "La Élite Académica" (RIESGO BAJO)
Tasa de Morosidad: 9.03%.

Volumen: 443 clientes (Un nicho pequeño).

¿Quiénes son? El 100% tiene un Máster. Además, son los que tienen el Ratio de Deuda más bajo de todos (0.45) y menos personas a cargo (0.38).

Insight de negocio: Son clientes muy cualificados, con alta educación financiera, que no se sobreendeudan. Son buenos pagadores.

🏆 Cluster 3: "Los Divorciados" (EL PERFIL MÁS SEGURO)
Tasa de Morosidad: 4.96% (¡Casi no impagan!).

Volumen: 826 clientes.

¿Quiénes son? El 100% son Divorciados.

Insight de negocio: Es el dato más sorprendente y valioso. Los divorciados (probablemente porque ya han pasado por liquidaciones de gananciales, pagan pensiones y controlan al milímetro su economía) son, con muchísima diferencia, los clientes más seguros a los que el banco les puede prestar dinero para una vivienda.

In [ ]:
# 5. FUNCIÓN ENTRENAMIENTO (CON TIEMPOS Y GAP)
def entrenar_modelo(
        nombre_modelo,
        modelo,
        param_grid,
        X_train, X_test,
        y_train, y_test,
        usar_smote=False,
        usar_pca=False,
        threshold=None 
    ):
        
        steps = [("scaler", StandardScaler())]

        if usar_smote:
            steps.append(("smote", SMOTE(random_state=42)))
        if usar_pca:
            steps.append(("pca", PCA(n_components=0.95, random_state=42)))

        steps.append(("model", modelo))
        pipe = ImbPipeline(steps)

        param_grid_pipeline = {f"model__{k}": v for k,v in param_grid.items()}
        recall_scorer = make_scorer(recall_score, pos_label=1)

        # 1. MEDIR TIEMPO DE ENTRENAMIENTO 
        start_train = time.time()
        
        grid = GridSearchCV(pipe, param_grid_pipeline, cv=3, scoring=recall_scorer, n_jobs=-1)
        grid.fit(X_train, y_train)
        
        end_train = time.time()
        train_time = end_train - start_train  # Tiempo en segundos

        # 2. CÁLCULO DE THRESHOLD DINÁMICO (EN TRAIN PARA EVITAR LEAKAGE) 
        y_proba_train = grid.best_estimator_.predict_proba(X_train)[:, 1]
        
        if threshold is None:
            precisions, recalls, thresholds = precision_recall_curve(y_train, y_proba_train)
            f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
            best_idx = np.argmax(f1_scores)
            best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        else:
            best_threshold = threshold
            
        y_pred_train = (y_proba_train >= best_threshold).astype(int)

        # 3. PREDICCIÓN EN TEST Y TIEMPOS
        start_pred = time.time()
        y_proba_test = grid.best_estimator_.predict_proba(X_test)[:, 1]
        y_pred_test = (y_proba_test >= best_threshold).astype(int)
        end_pred = time.time()
        
        prediction_time = end_pred - start_pred

        # CÁLCULO DE METRICAS EN TRAIN vs TEST (Para ver Overfitting)
        train_acc = accuracy_score(y_train, y_pred_train)
        test_acc = accuracy_score(y_test, y_pred_test)
        gap = (train_acc - test_acc) * 100 

        # 4. OTRAS MÉTRICAS TEST
        roc = roc_auc_score(y_test, y_proba_test)
        recall1 = recall_score(y_test, y_pred_test, pos_label=1)
        precision1 = precision_score(y_test, y_pred_test, pos_label=1, zero_division=0)

        print("="*60)
        print(f"{nombre_modelo} | SMOTE={usar_smote} | PCA={usar_pca} | THRESH={best_threshold:.4f}")
        print(f" Tiempo Train: {train_time:.2f}s | Tiempo Pred: {prediction_time:.4f}s")
        print(f" Acc Train: {train_acc:.4f} | Acc Test: {test_acc:.4f} | GAP: {gap:.2f}%")
        print(" ROC-AUC:", round(roc,4))
        print(" Recall (Impago):", round(recall1,4))

        return {
            "Modelo": nombre_modelo,
            "SMOTE": usar_smote,
            "PCA": usar_pca,
            "Threshold": best_threshold,
            "Train_Time_Sec": train_time,     
            "Pred_Time_Sec": prediction_time,  
            "Train_Accuracy": train_acc,      
            "Test_Accuracy": test_acc,         
            "Overfitting_Gap_Pct": gap,         
            "ROC_AUC": roc,
            "Recall_1": recall1,
            "Precision_1": precision1
        }

In [ ]:
# 6. DEFINIR MODELOS
modelos = {
    "LogReg": (LogisticRegression(max_iter=1000, class_weight="balanced"), {"C":[0.01,0.1,1]}),
    "RandomForest": (RandomForestClassifier(random_state=42, class_weight="balanced"), {"n_estimators":[100,200]}),
    "DecisionTree": (DecisionTreeClassifier(random_state=42, class_weight="balanced"), {"max_depth":[None,5,10]}),
    "AdaBoost": (AdaBoostClassifier(random_state=42), {"n_estimators":[50,100]}),
    "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=42, use_label_encoder=False),
                {"n_estimators":[100], "max_depth":[3,6]})
}

estimadores_base = [
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("dt", DecisionTreeClassifier(random_state=42)),
    ("nb", GaussianNB())
]

stacking = StackingClassifier(
    estimators=estimadores_base,
    final_estimator=LogisticRegression()
)

modelos["Stacking"] = (stacking, {"final_estimator__C":[0.1,1]})

# 7. EJECUCIÓN PARA TODAS LAS COMBINACIONES
combinaciones = [
    (False, False), # 1. Nada
    (True, False),  # 2. Solo SMOTE
    (False, True),  # 3. Solo PCA
    (True, True)    # 4. SMOTE + PCA
]

resultados_finales = []

for nombre, (modelo, grid) in modelos.items():
    for smote_flag, pca_flag in combinaciones:
        
        res = entrenar_modelo(
            nombre, modelo, grid,
            X_train, X_test,
            y_train, y_test,
            usar_smote=smote_flag,
            usar_pca=pca_flag,
            threshold=None 
        )
        
        resultados_finales.append(res)

df_resultados = pd.DataFrame(resultados_finales)

LogReg | SMOTE=False | PCA=False | THRESH=0.5399
 Tiempo Train: 0.22s | Tiempo Pred: 0.0040s
 Acc Train: 0.6513 | Acc Test: 0.6338 | GAP: 1.75%
 ROC-AUC: 0.6123
 Recall (Impago): 0.4776
LogReg | SMOTE=True | PCA=False | THRESH=0.5207
 Tiempo Train: 0.29s | Tiempo Pred: 0.0050s
 Acc Train: 0.6073 | Acc Test: 0.5969 | GAP: 1.05%
 ROC-AUC: 0.6086
 Recall (Impago): 0.5417
LogReg | SMOTE=False | PCA=True | THRESH=0.5134
 Tiempo Train: 0.19s | Tiempo Pred: 0.0050s
 Acc Train: 0.6000 | Acc Test: 0.5840 | GAP: 1.60%
 ROC-AUC: 0.6079
 Recall (Impago): 0.5545
LogReg | SMOTE=True | PCA=True | THRESH=0.4777
 Tiempo Train: 0.26s | Tiempo Pred: 0.0040s
 Acc Train: 0.5366 | Acc Test: 0.5280 | GAP: 0.85%
 ROC-AUC: 0.6071
 Recall (Impago): 0.6474
RandomForest | SMOTE=False | PCA=False | THRESH=0.5700
 Tiempo Train: 3.80s | Tiempo Pred: 0.1995s
 Acc Train: 0.9191 | Acc Test: 0.8206 | GAP: 9.85%
 ROC-AUC: 0.5327
 Recall (Impago): 0.0929
RandomForest | SMOTE=True | PCA=False | THRESH=0.4020
 Tiempo Train:

c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:41:19] WARNING: D:\bld\xgboost-split_1768313916136\work\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost | SMOTE=False | PCA=False | THRESH=0.1984
 Tiempo Train: 1.99s | Tiempo Pred: 0.0100s
 Acc Train: 0.8722 | Acc Test: 0.7938 | GAP: 7.84%
 ROC-AUC: 0.5744
 Recall (Impago): 0.1731


c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:41:20] WARNING: D:\bld\xgboost-split_1768313916136\work\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost | SMOTE=True | PCA=False | THRESH=0.3049
 Tiempo Train: 1.88s | Tiempo Pred: 0.0090s
 Acc Train: 0.8731 | Acc Test: 0.8070 | GAP: 6.60%
 ROC-AUC: 0.5722
 Recall (Impago): 0.1603


c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:41:22] WARNING: D:\bld\xgboost-split_1768313916136\work\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost | SMOTE=False | PCA=True | THRESH=0.2665
 Tiempo Train: 0.99s | Tiempo Pred: 0.0110s
 Acc Train: 0.9203 | Acc Test: 0.8045 | GAP: 11.58%
 ROC-AUC: 0.5691
 Recall (Impago): 0.141


c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:41:32] WARNING: D:\bld\xgboost-split_1768313916136\work\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost | SMOTE=True | PCA=True | THRESH=0.5657
 Tiempo Train: 10.13s | Tiempo Pred: 0.0080s
 Acc Train: 0.7837 | Acc Test: 0.7353 | GAP: 4.85%
 ROC-AUC: 0.578
 Recall (Impago): 0.2596
Stacking | SMOTE=False | PCA=False | THRESH=0.1574
 Tiempo Train: 21.55s | Tiempo Pred: 0.1109s
 Acc Train: 0.8629 | Acc Test: 0.8056 | GAP: 5.74%
 ROC-AUC: 0.5947
 Recall (Impago): 0.1603
Stacking | SMOTE=True | PCA=False | THRESH=0.3982
 Tiempo Train: 26.77s | Tiempo Pred: 0.1474s
 Acc Train: 0.9251 | Acc Test: 0.8034 | GAP: 12.17%
 ROC-AUC: 0.5353
 Recall (Impago): 0.1314
Stacking | SMOTE=False | PCA=True | THRESH=0.1399
 Tiempo Train: 51.06s | Tiempo Pred: 0.1313s
 Acc Train: 0.9077 | Acc Test: 0.8122 | GAP: 9.56%
 ROC-AUC: 0.5866
 Recall (Impago): 0.141
Stacking | SMOTE=True | PCA=True | THRESH=0.4000
 Tiempo Train: 93.00s | Tiempo Pred: 0.1427s
 Acc Train: 0.9247 | Acc Test: 0.7635 | GAP: 16.12%
 ROC-AUC: 0.5409
 Recall (Impago): 0.1571


In [ ]:
#Resultados
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)       

print("\n=========== RESULTADOS FINALES ===========")
print(df_resultados.sort_values("Recall_1", ascending=False).to_string(index=False))


=========== RESULTADOS FINALES ===========
      Modelo  SMOTE   PCA  Threshold  Train_Time_Sec  Pred_Time_Sec  Train_Accuracy  Test_Accuracy  Overfitting_Gap_Pct  ROC_AUC  Recall_1  Precision_1
DecisionTree   True  True   0.492632        0.694998       0.004002        0.458928       0.452215             0.671305 0.576539  0.730769     0.139024
DecisionTree   True False   0.421367        0.316840       0.004001        0.487978       0.492494            -0.451605 0.589210  0.701923     0.144841
      LogReg   True  True   0.477728        0.260912       0.004002        0.536556       0.528012             0.854388 0.607111  0.647436     0.146271
    AdaBoost   True  True   0.493609        3.814598       0.025058        0.550836       0.547785             0.305139 0.600972  0.634615     0.150114
    AdaBoost  False  True   0.320126        2.176746       0.020005        0.592457       0.578909             1.354815 0.616547  0.596154     0.153719
    AdaBoost   True False   0.478397        

Los 5 Mejores Modelos (Equilibrio Recall / Estabilidad / AUC)
#------------------- DecisionTree | SMOTE=True | PCA=False

Recall: 0.7019 (Detecta al ~70% de los impagos).

ROC-AUC: 0.5892

GAP: -0.45% (¡Excelente! No hay nada de overfitting).

Por qué es el #1: Tiene el mejor Recall de los modelos estables. Un GAP negativo leve indica que generaliza de maravilla en datos nuevos.

#------------------- DecisionTree | SMOTE=True | PCA=True

Recall: 0.7307

ROC-AUC: 0.5765

GAP: 0.67%

Por qué es el #2: Técnicamente tiene un Recall un poquito más alto que el #1, pero su ROC-AUC es peor. Aun así, su GAP inferior al 1% lo hace extremadamente sólido.

#------------------- LogReg | SMOTE=True | PCA=True

Recall: 0.6474 (Caza al ~65% de los morosos).

ROC-AUC: 0.6071 (Muy bueno, superior a los árboles).

GAP: 0.85%

Por qué es el #3: Es el "Golden Standard". La Regresión Logística es explicable, rápida (0.26s), no tiene overfitting y mantiene un AUC por encima de 0.60, lo que gusta mucho en banca.

#------------------- AdaBoost | SMOTE=True | PCA=True

Recall: 0.6346

ROC-AUC: 0.6009

GAP: 0.30%

Por qué es el #4: Un modelo de ensamblado robusto que consigue un gran equilibrio. Mantiene el AUC por encima del 0.60 con un overfitting prácticamente nulo.

#------------------- AdaBoost | SMOTE=False | PCA=True

Recall: 0.5961

ROC-AUC: 0.6165 (¡El AUC más alto del Top 5!)

GAP: 1.35%

Por qué es el #5: Si el banco prefiere equivocarse un poco menos con los clientes buenos (mejor AUC) a costa de dejar escapar a algún moroso más (baja el Recall a casi el 60%), esta sería la elección.

QUE HACER AHORA

A. Interpretabilidad (Lo que quiere el negocio)
Dado que la Regresión Logística o los Árboles (DecisionTree/AdaBoost) son tus mejores opciones reales, saca los Pesos (Coeficientes) o la Importancia de las Variables (Feature Importance).
A tu cliente (el banco) no le importa tanto que uses "Stacking" como que le digas: "Hemos descubierto que estar en el Cluster 2 y tener un ratio de deuda alto son los mayores detonantes del impago".

B. Matriz de Confusión Económica
Traduce tu mejor modelo a euros.

Asume un caso hipotético: Si el banco da 1.000 préstamos de 10.000€.

¿Cuánto dinero pierde el banco con el modelo actual vs tu modelo LogReg?

Calcula cuánto salvas al bloquear a ese 75% de morosos (True Positives), restando el beneficio que pierdes por los clientes buenos que bloqueaste por error (False Positives).

**GRAFICOS**

Este es el gráfico más "Pro" que puedes enseñar. Muestra en el eje Y cuánto acierta el modelo (Recall) y en el eje X cuánto "overfitting" tiene (el GAP).

Lo ideal es estar arriba a la izquierda (Mucho acierto, cero overfitting).

Verás a los modelos de Stacking y XGBoost perdidos por la derecha (mucho overfitting).

In [19]:
# GRÁFICO 1: RECALL vs OVERFITTING GAP
df_resultados['Config'] = df_resultados['Modelo'] + " | SMOTE:" + df_resultados['SMOTE'].astype(str) + " | PCA:" + df_resultados['PCA'].astype(str)

fig1 = px.scatter(
    df_resultados, 
    x="Overfitting_Gap_Pct", 
    y="Recall_1", 
    color="Modelo", 
    size="ROC_AUC", 
    hover_name="Config",
    hover_data={
        "Modelo": False,
        "Recall_1": ':.3f',
        "Overfitting_Gap_Pct": ':.2f',
        "ROC_AUC": ':.3f',
        "Threshold": ':.3f'
    },
    title="La Prueba del Algodón: Capacidad de Detección (Recall) vs Estabilidad (Overfitting)",
    labels={
        "Overfitting_Gap_Pct": "Brecha Train-Test (% Overfitting) ➔ Peor",
        "Recall_1": "Tasa de Detección de Morosos (Recall) ➔ Mejor"
    },
    template="plotly_white"
)

# Añadimos líneas de referencia (Lo ideal es estar en el cuadrante superior izquierdo)
fig1.add_vline(x=5, line_width=2, line_dash="dash", line_color="red", annotation_text="Límite Peligro Overfitting")
fig1.add_hline(y=0.60, line_width=2, line_dash="dash", line_color="green", annotation_text="Objetivo Mínimo Recall")

fig1.show()

In [21]:
# GRÁFICO 2: RANKING TOP 5 MODELOS ROBUSTOS
# 1. Filtramos para quitar los que tienen mucho overfitting (tramposos)
df_robustos = df_resultados[df_resultados['Overfitting_Gap_Pct'] < 5.0].copy()

# 2. Ordenamos por Recall y nos quedamos con los 10 mejores
df_top10 = df_robustos.sort_values(by="Recall_1", ascending=True).tail(5) 

fig2 = go.Figure()

# Barra del Recall (Detección de impagos)
fig2.add_trace(go.Bar(
    y=df_top10['Config'],
    x=df_top10['Recall_1'],
    name='Recall (Detección Impagos)',
    orientation='h',
    marker=dict(color='rgba(50, 171, 96, 0.7)', line=dict(color='rgba(50, 171, 96, 1.0)', width=1))
))

# Barra del ROC-AUC (Calidad matemática general)
fig2.add_trace(go.Bar(
    y=df_top10['Config'],
    x=df_top10['ROC_AUC'],
    name='ROC-AUC (Calidad General)',
    orientation='h',
    marker=dict(color='rgba(128, 114, 255, 0.7)', line=dict(color='rgba(128, 114, 255, 1.0)', width=1))
))

fig2.update_layout(
    title='Top 5 Modelos Estables (Overfitting < 5%)',
    barmode='group',
    xaxis_title='Puntuación (0 - 1)',
    yaxis_title='Configuración del Modelo',
    template="plotly_white",
    legend=dict(x=0.8, y=0.1) # Movemos la leyenda abajo a la derecha
)

fig2.show()